# Beyond Accuracy: Calibration Metrics for Clinical Decision Support

**Duration:** 25 minutes  
**Level:** Intermediate  
**Prerequisites:** Complete [Notebook 01: Scenario Instantiation](01_basics_scenario_instantiation.ipynb)

## Overview

**Calibration** asks: *Does the model know when it's uncertain?*

A well-calibrated model's confidence scores should match its empirical accuracy. When a CDSS predicts 80% confidence, it should be correct 80% of the time.

### What You'll Learn:

1. Expected Calibration Error (ECE)
2. Brier Score
3. Reliability curves (calibration plots)
4. Stratified calibration by risk tier
5. Multi-model calibration comparison

### Why This Matters:

Miscalibrated models can be dangerous in clinical settings:
- **Overconfident** models fail to escalate uncertain cases
- **Underconfident** models trigger unnecessary interventions
- **Risk-tier miscalibration** means different performance for high-risk vs. low-risk patients

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import BASICS-CDSS calibration module
from basics_cdss.metrics.calibration import (
    expected_calibration_error,
    brier_score,
    reliability_curve,
    stratified_calibration_metrics,
    calibration_summary
)
from basics_cdss.visualization.calibration_plots import (
    plot_reliability_diagram,
    plot_stratified_calibration,
    plot_calibration_comparison
)

print("✓ BASICS-CDSS calibration module loaded")

## 2. Generate Mock CDSS Data

We'll create realistic CDSS predictions for a triage task with three risk tiers.

In [ ]:
np.random.seed(42)
n_samples = 500

# Generate risk tiers (stratified distribution)
risk_tiers = np.random.choice(
    ['high', 'medium', 'low'],
    size=n_samples,
    p=[0.2, 0.3, 0.5]  # 20% high-risk, 30% medium, 50% low
)

# True labels (requires intervention: 1, defer: 0)
# High-risk patients more likely to need intervention
y_true = np.zeros(n_samples, dtype=int)
y_true[risk_tiers == 'high'] = np.random.binomial(1, 0.8, (risk_tiers == 'high').sum())
y_true[risk_tiers == 'medium'] = np.random.binomial(1, 0.4, (risk_tiers == 'medium').sum())
y_true[risk_tiers == 'low'] = np.random.binomial(1, 0.1, (risk_tiers == 'low').sum())

# Simulate CDSS predicted probabilities (somewhat correlated with truth)
# Model is miscalibrated: overconfident
base_probs = y_true * 0.7 + (1 - y_true) * 0.25 + np.random.normal(0, 0.15, n_samples)
base_probs = np.clip(base_probs, 0.01, 0.99)

# Make model overconfident (push toward extremes)
y_prob_overconfident = np.where(
    base_probs > 0.5,
    base_probs + 0.15,
    base_probs - 0.1
)
y_prob_overconfident = np.clip(y_prob_overconfident, 0.01, 0.99)

# Also create a well-calibrated model for comparison
y_prob_calibrated = base_probs + np.random.normal(0, 0.05, n_samples)
y_prob_calibrated = np.clip(y_prob_calibrated, 0.01, 0.99)

print(f"Generated {n_samples} CDSS predictions")
print(f"Risk tier distribution:")
print(pd.Series(risk_tiers).value_counts())
print(f"\nIntervention rate (ground truth): {y_true.mean():.1%}")
print(f"Average confidence (overconfident model): {y_prob_overconfident.mean():.3f}")
print(f"Average confidence (calibrated model): {y_prob_calibrated.mean():.3f}")

## 3. Expected Calibration Error (ECE)

ECE measures the average absolute difference between confidence and accuracy across probability bins.

$$
\text{ECE} = \sum_{m=1}^{M} \frac{|B_m|}{N} |\text{conf}(B_m) - \text{acc}(B_m)|
$$

where $B_m$ is the set of samples in bin $m$.

In [ ]:
# Compute ECE for both models
ece_overconfident = expected_calibration_error(y_true, y_prob_overconfident, n_bins=10)
ece_calibrated = expected_calibration_error(y_true, y_prob_calibrated, n_bins=10)

print("Expected Calibration Error (ECE)")
print("="*60)
print(f"Overconfident Model: {ece_overconfident:.4f}")
print(f"Calibrated Model:    {ece_calibrated:.4f}")
print(f"\nImprovement: {(ece_overconfident - ece_calibrated):.4f}")
print(f"Relative reduction: {((ece_overconfident - ece_calibrated) / ece_overconfident * 100):.1f}%")

print("\nInterpretation:")
print(f"  ECE < 0.05: Excellent calibration")
print(f"  ECE 0.05-0.10: Good calibration")
print(f"  ECE 0.10-0.15: Moderate miscalibration")
print(f"  ECE > 0.15: Poor calibration (unsafe for clinical use)")

## 4. Brier Score

Brier score measures mean squared error between predictions and outcomes:

$$
\text{BS} = \frac{1}{N} \sum_{i=1}^{N} (\hat{y}_i - y_i)^2
$$

In [ ]:
# Compute Brier scores
bs_overconfident = brier_score(y_true, y_prob_overconfident)
bs_calibrated = brier_score(y_true, y_prob_calibrated)

print("Brier Score")
print("="*60)
print(f"Overconfident Model: {bs_overconfident:.4f}")
print(f"Calibrated Model:    {bs_calibrated:.4f}")
print(f"\nLower is better (0 = perfect, 1 = worst)")
print(f"Improvement: {(bs_overconfident - bs_calibrated):.4f}")

## 5. Reliability Curves (Calibration Plots)

Reliability curves visualize calibration by plotting empirical accuracy against confidence.
Perfect calibration follows the diagonal (y=x).

In [ ]:
# Compute reliability curves
confs_overconf, accs_overconf, counts_overconf = reliability_curve(
    y_true, y_prob_overconfident, n_bins=10
)

confs_calib, accs_calib, counts_calib = reliability_curve(
    y_true, y_prob_calibrated, n_bins=10
)

# Plot overconfident model
fig1, ax1 = plot_reliability_diagram(
    confs_overconf, accs_overconf, counts_overconf,
    title="Overconfident Model: Calibration Analysis",
    color="#E63946"
)
plt.show()

# Plot calibrated model
fig2, ax2 = plot_reliability_diagram(
    confs_calib, accs_calib, counts_calib,
    title="Calibrated Model: Calibration Analysis",
    color="#06A77D"
)
plt.show()

print("✓ Reliability diagrams plotted")
print("\nObservations:")
print("  - Overconfident model: curve below diagonal (predicted confidence > actual accuracy)")
print("  - Calibrated model: curve closer to diagonal")

## 6. Stratified Calibration by Risk Tier

**Critical insight**: A model may be well-calibrated overall but miscalibrated for specific risk tiers.
We must check calibration separately for high-risk, medium-risk, and low-risk patients.

In [ ]:
# Compute stratified calibration metrics
stratified_metrics = stratified_calibration_metrics(
    y_true, y_prob_overconfident, risk_tiers, n_bins=10
)

print("Stratified Calibration Analysis (Overconfident Model)")
print("="*70)
print(f"{'Risk Tier':<15} {'ECE':<15} {'Brier Score':<15} {'Sample Count':<15}")
print("-"*70)

for tier, metrics in stratified_metrics.items():
    sample_count = (risk_tiers == tier).sum()
    print(f"{tier:<15} {metrics.ece:<15.4f} {metrics.brier_score:<15.4f} {sample_count:<15}")

print("\nKey Insight:")
print("  Check if ECE varies significantly across risk tiers.")
print("  Tier-specific miscalibration indicates differential performance.")

### 6.1 Visualize Stratified Calibration

In [ ]:
# Prepare data for stratified plot
calibration_by_tier = {
    tier: metrics.reliability_curve
    for tier, metrics in stratified_metrics.items()
}

fig, axes = plot_stratified_calibration(
    calibration_by_tier,
    title="Calibration by Risk Tier (Overconfident Model)"
)
plt.show()

print("✓ Stratified calibration plotted")
print("\nLook for:")
print("  - Different calibration patterns per tier")
print("  - High-risk tier: most critical to get right")
print("  - Systematic bias (curves consistently above/below diagonal)")

## 7. Multi-Model Calibration Comparison

Compare calibration across multiple CDSS models to identify the best-calibrated system.

In [ ]:
# Create a third model: underconfident
y_prob_underconfident = np.where(
    base_probs > 0.5,
    base_probs - 0.1,
    base_probs + 0.05
)
y_prob_underconfident = np.clip(y_prob_underconfident, 0.01, 0.99)

# Compute reliability curves for all models
models_calibration = {
    "Overconfident": reliability_curve(y_true, y_prob_overconfident, n_bins=10)[:2],
    "Well-Calibrated": reliability_curve(y_true, y_prob_calibrated, n_bins=10)[:2],
    "Underconfident": reliability_curve(y_true, y_prob_underconfident, n_bins=10)[:2],
}

# Plot comparison
fig, ax = plot_calibration_comparison(
    models_calibration,
    title="Multi-Model Calibration Comparison"
)
plt.show()

print("✓ Multi-model comparison plotted")

# Compute ECE for all models
print("\nModel Ranking by ECE:")
print("="*50)
model_eces = [
    ("Overconfident", expected_calibration_error(y_true, y_prob_overconfident)),
    ("Well-Calibrated", expected_calibration_error(y_true, y_prob_calibrated)),
    ("Underconfident", expected_calibration_error(y_true, y_prob_underconfident)),
]
model_eces.sort(key=lambda x: x[1])

for rank, (name, ece) in enumerate(model_eces, 1):
    print(f"{rank}. {name:<20} ECE = {ece:.4f}")

## 8. Comprehensive Calibration Summary

Use the `calibration_summary()` function for a complete calibration report.

In [ ]:
# Generate comprehensive summary
summary = calibration_summary(
    y_true,
    y_prob_overconfident,
    risk_tiers=risk_tiers,
    n_bins=10
)

print("Comprehensive Calibration Summary")
print("="*70)
print("\nOVERALL METRICS:")
print(f"  ECE:          {summary['overall']['ece']:.4f}")
print(f"  Brier Score:  {summary['overall']['brier_score']:.4f}")

print("\nSTRATIFIED BY RISK TIER:")
for tier, metrics in summary['by_risk_tier'].items():
    print(f"\n  {tier.upper()} Risk:")
    print(f"    ECE:          {metrics.ece:.4f}")
    print(f"    Brier Score:  {metrics.brier_score:.4f}")

print("\n✓ Comprehensive summary generated")

## 9. Clinical Interpretation Guide

### When to Trust CDSS Confidence Scores:

1. **ECE < 0.05**: Confidence scores are highly reliable
   - Safe to use for clinical decision-making
   - Clinicians can trust predicted probabilities

2. **ECE 0.05-0.10**: Good calibration with minor deviations
   - Generally reliable
   - Monitor for tier-specific miscalibration

3. **ECE > 0.10**: Poor calibration
   - Confidence scores may mislead clinicians
   - Consider recalibration (temperature scaling, Platt scaling)
   - Use with caution

### Red Flags:

- **High-risk tier miscalibration**: Most critical to address
- **Overconfidence in errors**: Model very confident when wrong
- **Extreme confidence bins**: All predictions near 0 or 1

## 10. Summary and Key Takeaways

### Core Concepts:

1. **Expected Calibration Error (ECE)**: Quantifies calibration quality (lower is better)
2. **Brier Score**: Mean squared error of probability predictions
3. **Reliability Curves**: Visual calibration assessment (diagonal = perfect)
4. **Stratified Calibration**: Different risk tiers may have different calibration
5. **Multi-Model Comparison**: Identify best-calibrated system

### Key API Functions:

```python
# Basic calibration metrics
ece = expected_calibration_error(y_true, y_prob, n_bins=10)
bs = brier_score(y_true, y_prob)
confs, accs, counts = reliability_curve(y_true, y_prob)

# Stratified analysis
stratified = stratified_calibration_metrics(y_true, y_prob, risk_tiers)

# Comprehensive summary
summary = calibration_summary(y_true, y_prob, risk_tiers=risk_tiers)
```

### Next Steps:

- **[Notebook 03](03_coverage_risk_selective_prediction.ipynb)**: Coverage-risk curves and selective prediction
- **[Notebook 04](04_harm_aware_evaluation.ipynb)**: Harm-aware metrics
- **[Notebook 05](05_end_to_end_pipeline.ipynb)**: Complete evaluation pipeline

### Clinical Impact:

Well-calibrated models enable:
- **Safe abstention**: Model knows when to defer to humans
- **Trusted predictions**: Clinicians can rely on confidence scores
- **Risk stratification**: Appropriate escalation based on uncertainty